In [1]:
import os
from elasticsearch import Elasticsearch
from rag_helper import RAGBase, INSTRUCTIONS
from ingest import load_labor_code_data
from google import genai
from dotenv import load_dotenv


# 1. Load the API key from your .env file
load_dotenv()
# 1. Connect to the local Docker container
es_client = Elasticsearch('http://localhost:9200')
index_name = "juris-maroc"

# 2. Define the Elasticsearch mapping (schema)
index_settings = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "content": {"type": "text"},
            "article_number": {"type": "keyword"}
        }
    }
}

# 3. Create the index and push your documents into it
es_client.indices.create(index=index_name, body=index_settings, ignore=[400])
documents = load_labor_code_data()

for doc in documents:
    es_client.index(index=index_name, document=doc)
print(f"Indexed {len(documents)} documents into Elasticsearch.")

# 4. Create the new class inheriting from RAGBase
class ElasticRAG(RAGBase):
    def __init__(self, es_client, index_name, llm_client, instructions):
        # Initialize the parent class without minsearch
        super().__init__(index=None, llm_client=llm_client, instructions=instructions)
        self.es_client = es_client
        self.index_name = index_name

    def search(self, query, num_results=5):
        # Elasticsearch specific query syntax
        search_query = {
            "size": num_results,
            "query": {
                "bool": {
                    "must": {
                        "multi_match": {
                            "query": query,
                            "fields": ["content^3", "article_number"], # Boosts content importance
                            "type": "best_fields"
                        }
                    }
                }
            }
        }

        response = self.es_client.search(index=self.index_name, body=search_query)

        # Extract the source documents from the ES JSON response
        results = [hit['_source'] for hit in response['hits']['hits']]
        return results

# 5. Test the new enterprise pipeline
client = genai.Client()
es_assistant = ElasticRAG(es_client=es_client, index_name=index_name, llm_client=client, instructions=INSTRUCTIONS)

answer = es_assistant.rag("Quels sont les droits d'une femme enceinte ?")
print(answer)

/tmp/ipykernel_15560/518760338.py:27: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.create(index=index_name, body=index_settings, ignore=[400])
/tmp/ipykernel_15560/518760338.py:27: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body=index_settings, ignore=[400])


Indexed 587 documents into Elasticsearch.


/tmp/ipykernel_15560/518760338.py:59: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = self.es_client.search(index=self.index_name, body=search_query)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Basé uniquement sur le contexte fourni, les droits et dispositions concernant la femme enceinte (et la femme en général) sont les suivants :

1. **Examen médical du travail** : Dans les entreprises soumises à l'obligation de disposer d'un service médical du travail, la femme enceinte a le droit (et l'obligation) de faire l'objet d'un examen médical par le médecin du travail, selon une fréquence dont ce dernier reste juge (*Article 327*).
2. **Non-discrimination et droits généraux** : En tant que femme, elle bénéficie de l'interdiction de toute discrimination fondée sur le sexe ou la situation conjugale (notamment concernant l'embauchage, le salaire, le licenciement, etc.), du droit de conclure un contrat de travail et du droit d'adhérer à un syndicat professionnel (*Article 9*).

*Note : Les autres droits spécifiques à la maternité (tels que le congé de maternité) ne figurent pas dans le contexte fourni.*


In [2]:
from google.genai import types

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search",
            description="Recherche dans la base de données du Code du Travail marocain pour trouver les articles pertinents.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="Le texte de la requête de recherche juridique."
                    )
                },
                required=["query"]
            )
        )
    ]
)

In [3]:
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_exponential

def calculate_gemini_flash_price(input_tokens: int, output_tokens: int) -> dict:
        # Standard pricing for Gemini 1.5/2.5 Flash
        INPUT_PRICE_PER_MILLION = 0.075
        OUTPUT_PRICE_PER_MILLION = 0.30

        input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
        output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
        total_cost = input_cost + output_cost

        return {
            "input_cost": input_cost,
            "output_cost": output_cost,
            "total_cost": total_cost,
        }



class AgenticElasticRAG:
    def __init__(self, es_client, index_name, llm_client, instructions):
        self.es_client = es_client
        self.index_name = index_name
        self.llm_client = llm_client
        self.instructions = instructions

        # 1. Initialize the Chat Session with your tool
        self.chat_session = self.llm_client.chats.create(
            model= "gemini-3.6-flash",
            config=types.GenerateContentConfig(
                system_instruction=self.instructions,
                tools=[self.search_juris_maroc], # We pass the method directly
                temperature=0.1 # Kept low for factual legal accuracy
            )
        )

    # 2. Define the tool with explicit Type Hints and a Docstring.
    # Gemini reads this metadata to understand HOW and WHEN to use the tool.
    def search_juris_maroc(self, query: str) -> str:
        """Recherche dans la base de données du Code du Travail marocain pour trouver les articles pertinents.

        Args:
            query: Le texte de la requête de recherche juridique.
        """
        print(f"\n[Outil Agent] L'agent a décidé de chercher : '{query}'")

        search_query = {
            "size": 5,
            "query": {
                "bool": {
                    "must": {
                        "multi_match": {
                            "query": query,
                            "fields": ["content^3", "article_number"],
                            "type": "best_fields"
                        }
                    }
                }
            }
        }

        response = self.es_client.search(index=self.index_name, **search_query)
        results = [hit['_source']['content'] for hit in response['hits']['hits']]

        # Return a single concatenated string for Gemini to read
        return "\n\n---\n\n".join(results)

    # 3. The execution method with your network retry shield
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def ask(self, user_message: str):
        # AFC handles the loop: it receives the message, triggers the tool if needed, and returns the final text.
        response = self.chat_session.send_message(user_message)
        # Extract token usage directly from the Gemini response metadata
        in_tokens = response.usage_metadata.prompt_token_count
        out_tokens = response.usage_metadata.candidates_token_count
        # Calculate cost
        cost_data = calculate_gemini_flash_price(in_tokens, out_tokens)
        print(f"📊 [FinOps] Tokens: {in_tokens} In | {out_tokens} Out")
        print(f"💰 [FinOps] Cost: ${cost_data['total_cost']:.8f}")

        return response.text

# --- Test the new Agent ---
agent = AgenticElasticRAG(
    es_client=es_client,
    index_name=index_name,
    llm_client=client,
    instructions=INSTRUCTIONS
)

# Ask a conversational question
print("Utilisateur: Bonjour !")
print(f"Agent: {agent.ask('Bonjour !')}")

# Ask a complex legal question to trigger the tool
print("\nUtilisateur: Peux-tu m'expliquer les conditions de la période d'essai ?")
final_answer = agent.ask("Peux-tu m'expliquer les conditions de la période d'essai ?")
print(f"\nAgent:\n{final_answer}")

Utilisateur: Bonjour !
📊 [FinOps] Tokens: 154 In | 19 Out
💰 [FinOps] Cost: $0.00001725
Agent: Bonjour ! Comment puis-je vous aider aujourd'hui concernant le droit du travail marocain ?

Utilisateur: Peux-tu m'expliquer les conditions de la période d'essai ?

[Outil Agent] L'agent a décidé de chercher : 'période d'essai'
📊 [FinOps] Tokens: 1447 In | 387 Out
💰 [FinOps] Cost: $0.00022462

Agent:
Selon le Code du Travail marocain (Articles 13 et 14), les conditions de la période d'essai sont définies comme suit :

### 1. Définition et préavis de rupture (Article 13)
* **Principe général :** La période d'essai permet à chaque partie de rompre volontairement le contrat de travail sans préavis ni indemnité.
* **Préavis obligatoire après une semaine :** Si le salarié a accompli au moins une semaine de travail, la rupture (non motivée par une faute grave) nécessite un préavis de :
  * **2 jours** si le salarié est payé à la journée, à la semaine ou à la quinzaine ;
  * **8 jours** s'il est payé

In [8]:
from google.genai import types

# 1. Initialize the raw conversation history (equivalent to OpenAI's 'messages' array)
messages = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text="Peux-tu m'expliquer les conditions de la période d'essai ?")]
    )
]

# 2. Force manual mode by explicitly disabling Automatic Function Calling
config = types.GenerateContentConfig(
    tools=[agent.search_juris_maroc], # Using the tool you defined earlier
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
)

print("Début de la boucle agentique...")

while True:
    # --- ETAPE A : Appel au LLM ---
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=messages,
        config=config
    )

    # IMPORTANT : Ajouter la réponse exacte du modèle à l'historique (qu'il s'agisse de texte ou d'une demande d'outil)
    messages.append(response.candidates[0].content)

    # --- ETAPE B : Vérification de la décision de l'Agent ---
    if not response.function_calls:
        # L'agent n'a pas appelé d'outil. C'est la réponse finale !
        print("\n✅ Réponse finale générée :")
        print(response.text)
        break # On sort de la boucle

    # --- ETAPE C : Exécution de l'Outil (équivalent de votre 'call.call_id') ---
    tool_call = response.function_calls[0]
    print(f"\n⚙️ [Boucle] L'agent a décidé d'utiliser l'outil : {tool_call.name}")
    print(f"⚙️ [Boucle] Mots-clés générés par l'agent : {tool_call.args}")

    if tool_call.name == "search_juris_maroc":
        query = tool_call.args["query"]

        # Vous exécutez la recherche Elasticsearch localement
        search_results = agent.search_juris_maroc(query)

        # --- ETAPE D : Injection des résultats ---
        # Équivalent de 'messages.append({"type": "function_call_output"...})'
        tool_response_part = types.Part.from_function_response(
            name=tool_call.name,
            response={"result": search_results}
        )

        # On ajoute le résultat dans l'historique avec le rôle "tool"
        messages.append(
            types.Content(role="user", parts=[tool_response_part])
        )
        print("⚙️ [Boucle] Données Elasticsearch injectées dans l'historique. L'agent réfléchit à nouveau...")

Début de la boucle agentique...

⚙️ [Boucle] L'agent a décidé d'utiliser l'outil : search_juris_maroc
⚙️ [Boucle] Mots-clés générés par l'agent : {'query': "période d'essai durée renouvellement rupture"}

[Outil Agent] L'agent a décidé de chercher : 'période d'essai durée renouvellement rupture'
⚙️ [Boucle] Données Elasticsearch injectées dans l'historique. L'agent réfléchit à nouveau...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 8h59m35.930763155s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See htt